<a href="https://colab.research.google.com/github/RolandoRobledo/Proyectos/blob/main/PIA_TEORIA_DE_LA_COMPUTACION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import random

def generar_instancia(n, m, p_min=1, p_max=100, semilla=None):
    """Genera una instancia aleatoria con n trabajos y m máquinas."""
    rng = random.Random(semilla)
    p = [rng.randint(p_min, p_max) for _ in range(n)]
    return {"n": n, "m": m, "p": p}


def instancia_peor_graham(m):
    """
    Instancia del peor caso para el algoritmo de Graham.
    m*(m-1) trabajos de tamaño 1 y un trabajo final de tamaño m.
    """
    p = [1] * (m * (m - 1)) + [m]
    return {"n": len(p), "m": m, "p": p}


def instancia_peor_lpt(m):
    """
    Instancia del peor caso para el algoritmo LPT (Largest Processing Time).
    Dos trabajos de tamaño k para k = 2m-1 hasta m+1, y tres de tamaño m.
    """
    p = []
    for k in range(2 * m - 1, m, -1):
        p += [k, k]
    p += [m, m, m]
    return {"n": len(p), "m": m, "p": p}


def cota_inferior(inst):
    """
    Calcula una cota inferior (Lower Bound) para el makespan óptimo:
    LB = max( trabajo más largo, techo(suma de tiempos / m) )
    """
    return max(max(inst["p"]), math.ceil(sum(inst["p"]) / inst["m"]))


if __name__ == "__main__":
    # Prueba con una instancia aleatoria de 6 trabajos y 3 máquinas
    ejemplo = generar_instancia(n=6, m=3)
    print("Instancia aleatoria:", ejemplo)
    print("Cota inferior (LB):", cota_inferior(ejemplo))

    # Prueba con la instancia del peor caso de Graham para m=3
    ej_graham = instancia_peor_graham(m=3)
    print("\nInstancia peor Graham (m=3):", ej_graham)
    print("Cota inferior (LB):", cota_inferior(ej_graham))

Instancia aleatoria: {'n': 6, 'm': 3, 'p': [14, 86, 40, 57, 37, 46]}
Cota inferior (LB): 94

Instancia peor Graham (m=3): {'n': 7, 'm': 3, 'p': [1, 1, 1, 1, 1, 1, 3]}
Cota inferior (LB): 3


In [ ]:
import time
import heapq
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

def graham_scheduling(jobs, m):
    machines = [0] * m
    heapq.heapify(machines)
    for job in jobs:
        min_load = heapq.heappop(machines)
        heapq.heappush(machines, min_load + job)
    return max(machines)

sizes = [5000, 15000, 30000, 50000, 75000, 100000]
instances_per_size = 5
m_machines = 10

_ = graham_scheduling([10, 20, 30], 2)

records_graham = []
exec_id = 1

for n in sizes:
    for rep in range(1, instances_per_size + 1):
        inst_name = f"Inst_{n}_{rep}"
        jobs = rng.integers(1, 1001, size=n).tolist()

        t0 = time.perf_counter()
        ms_graham = graham_scheduling(jobs, m_machines)
        time_graham = (time.perf_counter() - t0) * 1000.0

        records_graham.append({
            "#": exec_id,
            "Instancia": inst_name,
            "Tamaño": n,
            "Tiempo (ms)": round(time_graham, 4),
            "Makespan": ms_graham
        })
        exec_id += 1

df_graham = pd.DataFrame(records_graham)
print("=== TABLA DE EJECUCIONES: ALGORITMO DE GRAHAM ===")
print(df_graham.to_string(index=False))

=== TABLA DE EJECUCIONES: ALGORITMO DE GRAHAM ===
 #     Instancia  Tamaño  Tiempo (ms)  Makespan
 1   Inst_5000_1    5000       1.1999    249959
 2   Inst_5000_2    5000       1.2834    247733
 3   Inst_5000_3    5000       1.1542    249009
 4   Inst_5000_4    5000       1.2227    248815
 5   Inst_5000_5    5000       1.1331    252380
 6  Inst_15000_1   15000       3.6250    752756
 7  Inst_15000_2   15000       3.3819    752246
 8  Inst_15000_3   15000       3.7121    748568
 9  Inst_15000_4   15000       3.4019    756136
10  Inst_15000_5   15000       3.5533    750037
11  Inst_30000_1   30000       6.9466   1508264
12  Inst_30000_2   30000       7.1044   1504663
13  Inst_30000_3   30000       6.8226   1505510
14  Inst_30000_4   30000       7.1486   1502307
15  Inst_30000_5   30000       6.8186   1495530
16  Inst_50000_1   50000      11.6226   2499295
17  Inst_50000_2   50000      11.3886   2505347
18  Inst_50000_3   50000      11.7653   2491920
19  Inst_50000_4   50000      11.3395 

In [ ]:
import time
import heapq
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

def lpt_scheduling(jobs, m):
    sorted_jobs = sorted(jobs, reverse=True)
    machines = [0] * m
    heapq.heapify(machines)
    for job in sorted_jobs:
        min_load = heapq.heappop(machines)
        heapq.heappush(machines, min_load + job)
    return max(machines)

sizes = [5000, 15000, 30000, 50000, 75000, 100000]
instances_per_size = 5
m_machines = 10

_ = lpt_scheduling([10, 20, 30], 2)

records_lpt = []
exec_id = 1

for n in sizes:
    for rep in range(1, instances_per_size + 1):
        inst_name = f"Inst_{n}_{rep}"
        jobs = rng.integers(1, 1001, size=n).tolist()

        t0 = time.perf_counter()
        ms_lpt = lpt_scheduling(jobs, m_machines)
        time_lpt = (time.perf_counter() - t0) * 1000.0

        records_lpt.append({
            "#": exec_id,
            "Instancia": inst_name,
            "Tamaño": n,
            "Tiempo (ms)": round(time_lpt, 4),
            "Makespan": ms_lpt
        })
        exec_id += 1

df_lpt = pd.DataFrame(records_lpt)
print("=== TABLA DE EJECUCIONES: HEURÍSTICA LPT ===")
print(df_lpt.to_string(index=False))

=== TABLA DE EJECUCIONES: HEURÍSTICA LPT ===
 #     Instancia  Tamaño  Tiempo (ms)  Makespan
 1   Inst_5000_1    5000       1.8913    249570
 2   Inst_5000_2    5000       1.9747    247304
 3   Inst_5000_3    5000       2.5417    248634
 4   Inst_5000_4    5000       2.3237    248466
 5   Inst_5000_5    5000       2.1555    251968
 6  Inst_15000_1   15000       6.9290    752463
 7  Inst_15000_2   15000       6.4299    751834
 8  Inst_15000_3   15000       6.0101    748105
 9  Inst_15000_4   15000       5.6383    755680
10  Inst_15000_5   15000       5.7989    749550
11  Inst_30000_1   30000      11.6894   1507925
12  Inst_30000_2   30000      11.9794   1504307
13  Inst_30000_3   30000      11.5688   1505002
14  Inst_30000_4   30000      12.0956   1501834
15  Inst_30000_5   30000      11.5367   1495186
16  Inst_50000_1   50000      20.8632   2498836
17  Inst_50000_2   50000      20.0490   2504942
18  Inst_50000_3   50000      22.5149   2491477
19  Inst_50000_4   50000      19.8026   251